# Arastirma asistani

arxiv ve google news cekip kisa ozet aldim.


In [ ]:
import arxiv
import feedparser


### arxiv


In [ ]:
def fetch_arxiv_papers(query='hydrogen energy',max_results=3):
    client=arxiv.Client()
    search=arxiv.Search(query=query,max_results=max_results,sort_by=arxiv.SortCriterion.SubmittedDate)
    papers=[]
    for result in client.results(search):
        papers.append({'title':result.title.strip(),'summary':result.summary.strip(),'url':result.entry_id})
    return papers

fetch_arxiv_papers()


### Google News


In [ ]:
def fetch_google_news(query='hydrogen energy',max_articles=3):
    url='https://news.google.com/rss/search?q='+query.replace(' ','+')+'&hl=en-US&gl=US&ceid=US:en'
    feed=feedparser.parse(url)
    news_items=[]
    for entry in feed.entries[:max_articles]:
        news_items.append({'title':entry.title,'link':entry.link})
    return news_items

fetch_google_news()


### Ozet


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from IPython.display import Markdown
client=OpenAI(base_url='https://openrouter.ai/api/v1',api_key=os.getenv('OPENROUTER_API_KEY'))
def research_agent(query):
    papers=fetch_arxiv_papers(query)
    news=fetch_google_news(query)
    ctx='PAPERS:\n'+'\n'.join(p['title']+' - '+p['summary'][:200] for p in papers)
    ctx+='\nNEWS:\n'+'\n'.join(n['title'] for n in news)
    r=client.chat.completions.create(model='openai/gpt-oss-120b:free',messages=[{'role':'user','content':f'Research agent. ## Papers ## News ## Takeaway\n{ctx}\nQ:{query}'}])
    return Markdown(r.choices[0].message.content)
research_agent('hydrogen energy')


### Sonuc

Makale basliklari + haberler tek ozet halinde geldi.
